# Bank Marketing – Model Training & Evaluation
**Target:** `y` (client subscribed to a term deposit: yes=1 / no=0).

Workflow (test set stays untouched until the very last step)
1. Load the train/test split produced by the preprocessing notebook (`train_raw.csv`, `test_raw.csv`).
2. Build full pipelines = preprocessing + model, so scaling/encoding are re-fitted inside every CV fold.
3. Compare models with stratified 5-fold CV on the **training set** (ROC-AUC, PR-AUC, F1, recall, precision).
4. Tune the best model on the training set only.
5. Choose the decision threshold from out-of-fold **training** predictions.
6. Evaluate **once** on the test set and visualise.

`duration` is excluded (known only after the call -> leakage), so this is a *pre-call* model.

## 0. Imports and configuration

In [ ]:
import os
import json
import time
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, average_precision_score, balanced_accuracy_score,
                             ConfusionMatrixDisplay, confusion_matrix, f1_score,
                             matthews_corrcoef, precision_recall_curve, precision_score,
                             recall_score, roc_auc_score, roc_curve, classification_report)
from sklearn.model_selection import (RandomizedSearchCV, StratifiedKFold, cross_validate,
                                     cross_val_predict)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
pd.set_option("display.width", 200)

RANDOM_STATE = 42
INCLUDE_DURATION = False                 # keep False for a deployable pre-call model
DATA_DIR = os.environ.get("DATA_DIR", "processed_data")
OUT_DIR = os.environ.get("OUT_DIR", "model_outputs")
os.makedirs(OUT_DIR, exist_ok=True)
TARGET, TARGET_NUM = "y", "y_encoded"

## 1. Load the split from preprocessing
In Colab, upload the `processed_data` folder (or at least `train_raw.csv` and `test_raw.csv`).

In [ ]:
def locate(name):
    for d in [DATA_DIR, "/content/processed_data", "."]:
        p = os.path.join(d, name)
        if os.path.exists(p):
            return p
    try:
        from google.colab import files
        print(f"Upload {name} ...")
        files.upload()
        return name
    except ImportError:
        raise FileNotFoundError(f"{name} not found. Run the preprocessing notebook first.")


train_df = pd.read_csv(locate("train_raw.csv"))
test_df = pd.read_csv(locate("test_raw.csv"))

drop_cols = [TARGET, TARGET_NUM] + ([] if INCLUDE_DURATION else ["duration"])
X_train, y_train = train_df.drop(columns=drop_cols), train_df[TARGET_NUM]
X_test, y_test = test_df.drop(columns=drop_cols), test_df[TARGET_NUM]   # not touched until Step 6

print("Target variable      :", TARGET, "-> numeric:", TARGET_NUM, "(yes=1, no=0)")
print("X_train / X_test     :", X_train.shape, X_test.shape)
print("Features used        :", list(X_train.columns))
print("Train 'yes' share    : %.4f" % y_train.mean())

## 2. Preprocessing + model pipelines
Same preprocessing as before: median-impute -> winsorise at train 1%/99% -> standard-scale numerics,
0/1 for binary columns, one-hot for the rest. Because it lives inside the Pipeline,
cross-validation re-fits it on each training fold only (no leakage into validation folds).

In [ ]:
class QuantileClipper(BaseEstimator, TransformerMixin):
    """Winsorise each column to quantile limits learned in fit() (train data only)."""

    def __init__(self, lower=0.01, upper=0.99):
        self.lower = lower
        self.upper = upper

    def fit(self, X, y=None):
        X = np.asarray(X, dtype=float)
        self.lower_ = np.quantile(X, self.lower, axis=0)
        self.upper_ = np.quantile(X, self.upper, axis=0)
        return self

    def transform(self, X):
        return np.clip(np.asarray(X, dtype=float), self.lower_, self.upper_)

    def get_feature_names_out(self, input_features=None):
        return np.asarray(input_features, dtype=object)


flag_cols = ["previously_contacted"]
non_numeric = [c for c in X_train.columns if not pd.api.types.is_numeric_dtype(X_train[c])]
binary_cols = [c for c in non_numeric if set(X_train[c].unique()) <= {"yes", "no"}]
cat_cols = [c for c in non_numeric if c not in binary_cols]
num_cols = [c for c in X_train.columns if c not in non_numeric and c not in flag_cols]


def make_preprocessor():
    return ColumnTransformer([
        ("num", Pipeline([("impute", SimpleImputer(strategy="median")),
                          ("clip", QuantileClipper(0.01, 0.99)),
                          ("scale", StandardScaler())]), num_cols),
        ("bin", Pipeline([("impute", SimpleImputer(strategy="most_frequent")),
                          ("encode", OrdinalEncoder(categories=[["no", "yes"]] * len(binary_cols),
                                                    handle_unknown="use_encoded_value",
                                                    unknown_value=-1))]), binary_cols),
        ("flag", "passthrough", flag_cols),
        ("cat", Pipeline([("impute", SimpleImputer(strategy="constant", fill_value="unknown")),
                          ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))]),
         cat_cols),
    ], verbose_feature_names_out=False)


def make_pipeline(model):
    return Pipeline([("prep", make_preprocessor()), ("model", model)])


# Imbalance is handled with class weights (no resampling needed -> nothing can leak through SMOTE).
candidates = {
    "Logistic Regression": make_pipeline(LogisticRegression(
        class_weight="balanced", max_iter=2000, random_state=RANDOM_STATE)),
    "Random Forest": make_pipeline(RandomForestClassifier(
        n_estimators=200, min_samples_leaf=5, class_weight="balanced_subsample",
        n_jobs=-1, random_state=RANDOM_STATE)),
    "Hist Gradient Boosting": make_pipeline(HistGradientBoostingClassifier(
        class_weight="balanced", random_state=RANDOM_STATE)),
    # Neural network: 2 hidden layers (32 -> 16 ReLU), Adam, L2 penalty, early stopping (see src/neural.py for the 5-network ensemble)
    "Neural Network (MLP)": make_pipeline(MLPClassifier(
        hidden_layer_sizes=(32, 16), alpha=0.1, early_stopping=True, max_iter=200, random_state=RANDOM_STATE)),
}
baseline = make_pipeline(DummyClassifier(strategy="prior"))

## 3. Compare models with stratified 5-fold CV (training data only)
Metrics are reported for the positive class ("yes"). PR-AUC is the main selection metric
because the data are imbalanced (~11.7% "yes"); accuracy alone would be misleading.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
scoring = {"roc_auc": "roc_auc", "pr_auc": "average_precision",
           "f1": "f1", "recall": "recall", "precision": "precision", "accuracy": "accuracy"}

cv_rows = []
for name, pipe in {"Baseline (always 'no')": baseline, **candidates}.items():
    t0 = time.time()
    res = cross_validate(pipe, X_train, y_train, cv=cv, scoring=scoring, n_jobs=1)
    row = {"model": name, "fit_time_s": round(time.time() - t0, 1)}
    for m in scoring:
        row[f"{m}_mean"] = res[f"test_{m}"].mean()
        row[f"{m}_std"] = res[f"test_{m}"].std()
    cv_rows.append(row)
    print(f"{name:25s} PR-AUC={row['pr_auc_mean']:.4f}  ROC-AUC={row['roc_auc_mean']:.4f}  "
          f"F1={row['f1_mean']:.4f}  recall={row['recall_mean']:.4f}")

cv_results = pd.DataFrame(cv_rows).set_index("model")
cv_results.round(4).to_csv(os.path.join(OUT_DIR, "cv_model_comparison.csv"))
best_name = cv_results.drop(index="Baseline (always 'no')")["pr_auc_mean"].idxmax()
print("\nBest model by CV PR-AUC:", best_name)

## 4. Tune the selected model (training data only)
Randomised search with 3-fold CV, optimising PR-AUC. The test set plays no part in this.

In [ ]:
param_spaces = {
    "Hist Gradient Boosting": {
        "model__learning_rate": [0.03, 0.05, 0.1],
        "model__max_iter": [100, 200, 300],
        "model__max_leaf_nodes": [15, 31, 63],
        "model__min_samples_leaf": [20, 50, 100],
        "model__l2_regularization": [0.0, 0.1, 1.0],
    },
    "Random Forest": {
        "model__n_estimators": [200, 300],
        "model__max_depth": [8, 12, 20, None],
        "model__min_samples_leaf": [3, 5, 10, 20],
        "model__max_features": ["sqrt", 0.3],
    },
    "Logistic Regression": {"model__C": [0.01, 0.1, 1, 10]},
}
search = RandomizedSearchCV(
    candidates[best_name], param_spaces[best_name],
    n_iter=min(10, int(np.prod([len(v) for v in param_spaces[best_name].values()]))),
    scoring="average_precision",
    cv=StratifiedKFold(3, shuffle=True, random_state=RANDOM_STATE),
    random_state=RANDOM_STATE, n_jobs=1, refit=True)
search.fit(X_train, y_train)
best_model = search.best_estimator_
print("Best params :", {k.replace("model__", ""): v for k, v in search.best_params_.items()})
print("CV PR-AUC   : %.4f" % search.best_score_)

## 5. Choose the decision threshold (from out-of-fold *training* predictions)
With balanced class weights the 0.5 cut-off is rarely the best trade-off. The threshold that maximises F1
is picked from cross-validated predictions on the training set, then frozen before touching the test set.

In [ ]:
oof_proba = cross_val_predict(best_model, X_train, y_train, cv=cv, method="predict_proba")[:, 1]
prec, rec, thr = precision_recall_curve(y_train, oof_proba)
f1_curve = 2 * prec[:-1] * rec[:-1] / np.clip(prec[:-1] + rec[:-1], 1e-12, None)
THRESHOLD = float(thr[np.argmax(f1_curve)])
print("Chosen threshold (max OOF F1 on train): %.3f  -> OOF F1 = %.4f" % (THRESHOLD, f1_curve.max()))

## 6. Final evaluation on the untouched test set (done once)

In [ ]:
test_proba = best_model.predict_proba(X_test)[:, 1]


def metrics_at(threshold):
    pred = (test_proba >= threshold).astype(int)
    return {
        "threshold": round(threshold, 3),
        "accuracy": accuracy_score(y_test, pred),
        "balanced_accuracy": balanced_accuracy_score(y_test, pred),
        "precision": precision_score(y_test, pred, zero_division=0),
        "recall": recall_score(y_test, pred),
        "f1": f1_score(y_test, pred),
        "mcc": matthews_corrcoef(y_test, pred),
        "roc_auc": roc_auc_score(y_test, test_proba),     # threshold-free
        "pr_auc": average_precision_score(y_test, test_proba),   # threshold-free
    }


base_pred = baseline.fit(X_train, y_train).predict(X_test)
test_results = pd.DataFrame({
    "Baseline (always 'no')": {"threshold": np.nan, "accuracy": accuracy_score(y_test, base_pred),
                               "balanced_accuracy": balanced_accuracy_score(y_test, base_pred),
                               "precision": 0.0, "recall": 0.0, "f1": 0.0, "mcc": 0.0,
                               "roc_auc": 0.5, "pr_auc": float(y_test.mean())},
    f"{best_name} @0.50": metrics_at(0.5),
    f"{best_name} @{THRESHOLD:.2f} (tuned)": metrics_at(THRESHOLD),
}).T
print(test_results.round(4).to_string())
test_results.round(4).to_csv(os.path.join(OUT_DIR, "test_metrics.csv"))

final_pred = (test_proba >= THRESHOLD).astype(int)
print("\nClassification report (tuned threshold):")
print(classification_report(y_test, final_pred, target_names=["no", "yes"], digits=3))
cm = confusion_matrix(y_test, final_pred)
print("Confusion matrix [[TN FP],[FN TP]]:\n", cm)

## 7. Visualisations

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 4.8))

# (a) CV comparison
plot_df = cv_results.drop(index="Baseline (always 'no')")[["roc_auc_mean", "pr_auc_mean", "f1_mean", "recall_mean"]]
plot_df.columns = ["ROC-AUC", "PR-AUC", "F1", "Recall"]
plot_df.plot.bar(ax=axes[0], rot=15, colormap="viridis")
axes[0].set_title("5-fold CV on training data")
axes[0].set_ylim(0, 1)
axes[0].legend(loc="lower right", fontsize=8)

# (b) ROC curve
fpr, tpr, _ = roc_curve(y_test, test_proba)
axes[1].plot(fpr, tpr, label=f"AUC = {roc_auc_score(y_test, test_proba):.3f}")
axes[1].plot([0, 1], [0, 1], "k--", alpha=0.5)
axes[1].set(xlabel="False positive rate", ylabel="True positive rate", title="ROC curve (test)")
axes[1].legend()

# (c) Precision-Recall curve
p, r, _ = precision_recall_curve(y_test, test_proba)
axes[2].plot(r, p, label=f"PR-AUC = {average_precision_score(y_test, test_proba):.3f}")
axes[2].axhline(y_test.mean(), color="k", ls="--", alpha=0.5, label="No-skill (class share)")
axes[2].set(xlabel="Recall", ylabel="Precision", title="Precision-Recall curve (test)")
axes[2].legend()
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "01_cv_roc_pr.png"), dpi=150)
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
for ax, (title, pred) in zip(axes, [("threshold 0.50", (test_proba >= 0.5).astype(int)),
                                     (f"tuned threshold {THRESHOLD:.2f}", final_pred)]):
    ConfusionMatrixDisplay(confusion_matrix(y_test, pred), display_labels=["no", "yes"]).plot(
        ax=ax, cmap="Blues", colorbar=False, values_format="d")
    ax.set_title(f"Confusion matrix – {title}")
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "02_confusion_matrices.png"), dpi=150)
plt.show()

In [ ]:
# Which raw features matter? Permutation importance (interpretation only – not used for any decision)
perm = permutation_importance(best_model, X_test, y_test, scoring="average_precision",
                              n_repeats=5, random_state=RANDOM_STATE, n_jobs=1)
imp = pd.Series(perm.importances_mean, index=X_test.columns).sort_values()
imp_std = pd.Series(perm.importances_std, index=X_test.columns)[imp.index]
fig, ax = plt.subplots(figsize=(7, 5))
ax.barh(imp.index, imp.values, xerr=imp_std.values, color="#4C72B0")
ax.set_xlabel("Drop in PR-AUC when the feature is shuffled")
ax.set_title("Permutation importance (test set)")
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "03_permutation_importance.png"), dpi=150)
plt.show()
print(imp.sort_values(ascending=False).round(4).head(8))

## 8. Save the model and a few example predictions

In [ ]:
joblib.dump({"pipeline": best_model, "threshold": THRESHOLD, "features": list(X_train.columns),
             "model_name": best_name}, os.path.join(OUT_DIR, "final_model.joblib"))
with open(os.path.join(OUT_DIR, "summary.json"), "w") as f:
    json.dump({"target": TARGET, "model": best_name,
               "best_params": {k.replace("model__", ""): (v if v is None or isinstance(v, (int, float, str)) else str(v))
                               for k, v in search.best_params_.items()},
               "threshold": THRESHOLD, "include_duration": INCLUDE_DURATION,
               "test_metrics_tuned": {k: float(v) for k, v in test_results.iloc[-1].items()}}, f, indent=2)

example = X_test.head(5).copy()
example["P(subscribe)"] = best_model.predict_proba(example)[:, 1].round(3)
example["prediction"] = np.where(example["P(subscribe)"] >= THRESHOLD, "yes", "no")
example["actual"] = test_df.loc[example.index, TARGET].values
print(example[["age", "job", "balance", "poutcome", "P(subscribe)", "prediction", "actual"]])
print("\nSaved to:", os.path.abspath(OUT_DIR), "->", sorted(os.listdir(OUT_DIR)))
print("To reuse: m = joblib.load('final_model.joblib'); "
      "m['pipeline'].predict_proba(new_df[m['features']])[:,1] >= m['threshold']")